# Week 14 — Recommendation Systems
### Building a Content-Based Movie Recommendation Engine
**Next Gen Software Hub Pvt. Ltd. — AI & ML Internship**

---
### Objectives:
1. Load and inspect the movie dataset.
2. Create a unified metadata 'content soup'.
3. Compute TF-IDF matrix and Cosine Similarity scores.
4. Implement the recommendation ranking engine.
5. Evaluate recommendations across multiple genres.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
df = pd.read_csv('datasets/movies_recommendation_data.csv')
print(f'Total Movies Loaded: {len(df)}')
df.head()

## 1. Feature Engineering: Content Soup Construction
Combine text attributes (Genre, Director, Keywords) to capture the thematic essence of each title.

In [ ]:
# Clean and combine attributes
df['Director_Clean'] = df['Director'].str.replace(' ', '')
df['Content_Soup'] = (
    df['Genre'].str.lower() + ' ' +
    df['Director_Clean'].str.lower() + ' ' +
    df['Keywords'].str.lower()
)
print('Sample Content Soup for movie #1:')
print(df['Content_Soup'].iloc[0])

## 2. Vectorization & Cosine Similarity Matrix

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['Content_Soup'])
print(f'TF-IDF Matrix Shape: {tfidf_matrix.shape}')

# Compute pairwise cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f'Cosine Similarity Matrix Shape: {cosine_sim.shape}')

## 3. Recommendation Function Implementation

In [ ]:
indices = pd.Series(df.index, index=df['Title'].str.lower()).drop_duplicates()

def recommend_movies(title, top_n=5):
    title_clean = title.strip().lower()
    if title_clean not in indices:
        return f"Movie '{title}' not found in database."
    
    idx = indices[title_clean]
    sim_scores = list(enumerate(cosine_sim[idx]))
    # Sort by similarity descending, excluding the query movie itself
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    scores = [round(i[1] * 100, 2) for i in sim_scores]
    
    results = df.iloc[movie_indices][['Title', 'Genre', 'Director', 'Rating']].copy()
    results['Similarity_Score_%'] = scores
    return results

print('Recommendation engine ready!')

## 4. Test Sample Recommendations

In [ ]:
print('=== Recommendations for: Inception ===')
display(recommend_movies('Inception', top_n=5))

print('\n=== Recommendations for: Toy Story ===')
display(recommend_movies('Toy Story', top_n=4))

print('\n=== Recommendations for: Fight Club ===')
display(recommend_movies('Fight Club', top_n=4))